 🌐 Create Study & Mission Nodes in the SPOKE‑GeneLab Knowledge Graph

This notebook reads the GeneLab dataset manifest file, extracts mission‑ and study‑level metadata, and writes Neo4j node and relationship files for integration into the SPOKE‑GeneLab Knowledge Graph via the `genelab_utils` package.

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [1]:
import pandas as pd
import genelab_utils as gl

In [2]:
pd.set_option('display.max_rows', None)  # Shows all rows
pd.set_option('display.max_colwidth', None)  # Shows full content of each cell

## Setup Environment Variables
Edit `../.env` to configure the environment.    

In [4]:
# Node and relationship directory paths
node_dir, rel_dir = gl.setup_environment()

Environment setup for KG version: v0.3.1


## Validate the KG Metadata Files in the `../kg/v#.#.#/metadata` Directory

In [5]:
gl.validate_kg_metadata()

Metadata files passed the check!


## Get Info about available Datasets

In [6]:
MANIFEST_PATH = "../data/manifest.csv" # file with dataset info

In [7]:
manifest = pd.read_csv(MANIFEST_PATH).fillna("")
manifest.head()

,identifier,technology,measurement,assay_name,taxonomy,material,host_organism,host_strain,filename,url,differential_analysis_method,organism,host_taxonomy
0,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_Affymetrix,7227,Whole Organism,,,GLDS-1_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-1/download?source=datamanager&file=GLDS-1_array_differential_expression.csv,DESeq2,Drosophila melanogaster,
1,OSD-3,DNA microarray,transcription profiling,OSD-3_transcription-profiling_dna-microarray_affymetrix,7227,Whole Organism,,,GLDS-3_array_differential_expression_GLmicroarray.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-3/download?source=datamanager&file=GLDS-3_array_differential_expression_GLmicroarray.csv,DESeq2,Drosophila melanogaster,
2,OSD-4,DNA microarray,transcription profiling,OSD-4_transcription-profiling_dna-microarray_affymetrix,10090,thymus,,,GLDS-4_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-4/download?source=datamanager&file=GLDS-4_array_differential_expression.csv,DESeq2,Mus musculus,
3,OSD-6,DNA microarray,transcription profiling,OSD-6_transcription-profiling_dna-microarray_affymetrix,10116,Cells,,,GLDS-6_array_differential_expression_GLmicroarray.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-6/download?source=datamanager&file=GLDS-6_array_differential_expression_GLmicroarray.csv,DESeq2,Rattus norvegicus,
4,OSD-13,DNA microarray,transcription profiling,OSD-13_transcription-profiling_dna-microarray_Affymetrix,9606,Primary T Cells,,,GLDS-13_array_differential_expression_GLmicroarray.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-13/download?source=datamanager&file=GLDS-13_array_differential_expression_GLmicroarray.csv,DESeq2,Homo sapiens,


## Get GeneLab Mission and Study Metadata

In [8]:
metadata = gl.get_metadata(manifest)
metadata.head()

,identifier,project_type,project_title,description,taxonomy,organism,host_organism,host_taxonomy,host_strain,flight_program,space_program,mission_id,name,start_date,end_date
0,OSD-1,Spaceflight Study,Drosophila as a model of immune function in conditions of altered gravitational force,"Space travel presents unlimited opportunities for exploration and discovery, but requires a more complete understanding of the immunological consequences of long-term exposure to the conditions of spaceflight. To understand these consequences better and to contribute to design of effective countermeasures, we used the Drosophila model to compare innate immune responses to bacteria and fungi in flies that were either raised on earth or in outer space aboard the NASA Space Shuttle Discovery (STS-121). Microarrays were used to characterize changes in gene expression that occur in response to infection by bacteria and fungus in drosophila that were either hatched and raised in outer space (microgravity) or on earth (normal gravity). Whole Oregon R strain drosophila melanogaster fruit flies either raised on earth or in space that were (1) uninfected, (2) infected with bacteria (Escherichia coli), or (3) infected with fungus (Beauveria bassiana) were used for RNA extraction and hybridization on Affymetrix microarrays.",7227,Drosophila melanogaster,,,,Space Transportation System (STS),NASA,STS-121,STS-121,2006-07-04,2006-07-17
1,OSD-3,Spaceflight Study,"Fungal Pathogenesis, Tumorigenesis, and Effects of Host Immunity in Space","Gene expression levels were determined in 3rd instar and adult Drosophila melanogaster reared during spaceflight, to elucidate the genetic and molecular mechanisms underpinning the effects of microgravity on the immune system. The goal was to validate the Drosophila model for understanding alterations of innate immune responses in humans due to spaceflight. Five containers of flies, with ten female and five male fruit flies in each container, were housed and bred on the space shuttle (average orbit altitude of 330.35 km) for 12 days and 18.5 hours, with a new generation reared in microgravity. RNA was extracted on the day of shuttle landing from whole body animals (3rd instar larvae and adults), hybridized to Drosophila 2.0 Affymetrix genome arrays, and the expression level of all genes was normalized against the gene expression level from the corresponding developmental stage animals raised on ground. Spaceflight altered the expression of larval genes involved in the maturation of plasmatocytes (macrophages) and their phagocytic response, as well as the level of constitutive expression of pattern recognition receptors and opsonins that specifically recognize bacteria, and of lysozymes, antimicrobial peptide pathway and immune stress genes, hallmarks of humoral immunity. Larval microarrays (FL 6 samples) are based on RNA extracted from 6 independent sets of 50 mid 3rd instar larvae reared in microgravity and collected on the day of landing after 12 days and 18.5 hours on the space shuttle and the same number of control larvae raised on ground (GL 6 samples). Adults microarrays (F1 3 samples) are based on RNA from 3 sets of 20 adult females each, that emerged during spaceflight and within 4 hours of landing and the same number of adult females from the corresponding ground control containers (G1 3 samples).",7227,Drosophila melanogaster,,,,Space Transportation System (STS),NASA,STS-121,STS-121,2006-07-04,2006-07-17
2,OSD-4,Spaceflight Study,The Effects of Vector-Averaged Gravity on the Development of T cells,"Microarray Analysis of Space-flown Murine Thymus Tissue Reveals Changes in Gene Expression Regulating Stress and Glucocorticoid Receptors. We used microarrays to detail the gene expression of space-flown thymic tissue and identified distinct classes of up-regulated genes during this process. We report here microarray gene expression analysis in young adult C57BL/6NTac mice at 8 weeks of age after exposure to spaceflight aboa

## Create Mission Nodes

In [9]:
missions = metadata[["mission_id", "name", "flight_program", "space_program", "start_date", "end_date"]]
missions = missions[missions["name"] != ""].copy()
missions.rename(columns={"mission_id": "identifier"}, inplace=True)

In [10]:
mission_nodes = gl.save_dataframe_to_kg(missions, 'Mission', node_dir)
print(f"Number of Mission nodes: {mission_nodes.shape[0]}")
mission_nodes.head()

Number of Mission nodes: 43


,identifier,name,flight_program,space_program,start_date,end_date
0,STS-121,STS-121,Space Transportation System (STS),NASA,2006-07-04,2006-07-17
4,Expedition-14,Expedition 14,International Space Station (ISS),NASA,2006-09-18,2007-04-21
6,STS-135,STS-135,Space Transportation System (STS),NASA,2011-07-08,2011-07-21
9,Soyuz-TMA-2,Soyuz TMA-2,International Space Station (ISS),ESA,2003-04-26,2003-10-28
10,Soyuz-TMA-3,Soyuz TMA-3,International Space Station (ISS),ESA,2003-10-18,2004-04-30


## Create Study Nodes

In [11]:
studies = metadata[["identifier", "project_title", "project_type", "description", "organism", "taxonomy", "host_organism", "host_taxonomy", "host_strain"]].copy()
studies["name"] = studies["identifier"]
studies = studies[["identifier", "name", "project_title", "project_type", "description", "organism", "taxonomy", "host_organism", "host_taxonomy", "host_strain"]]

In [12]:
study_nodes = gl.save_dataframe_to_kg(studies, 'Study', node_dir)
print(f"Number of Study nodes: {study_nodes.shape[0]}")
study_nodes.head()

Number of Study nodes: 227


,identifier,name,project_title,project_type,description,organism,taxonomy,host_organism,host_taxonomy,host_strain
0,OSD-1,OSD-1,Drosophila as a model of immune function in conditions of altered gravitational force,Spaceflight Study,"Space travel presents unlimited opportunities for exploration and discovery, but requires a more complete understanding of the immunological consequences of long-term exposure to the conditions of spaceflight. To understand these consequences better and to contribute to design of effective countermeasures, we used the Drosophila model to compare innate immune responses to bacteria and fungi in flies that were either raised on earth or in outer space aboard the NASA Space Shuttle Discovery (STS-121). Microarrays were used to characterize changes in gene expression that occur in response to infection by bacteria and fungus in drosophila that were either hatched and raised in outer space (microgravity) or on earth (normal gravity). Whole Oregon R strain drosophila melanogaster fruit flies either raised on earth or in space that were (1) uninfected, (2) infected with bacteria (Escherichia coli), or (3) infected with fungus (Beauveria bassiana) were used for RNA extraction and hybridization on Affymetrix microarrays.",Drosophila melanogaster,7227,,,
1,OSD-3,OSD-3,"Fungal Pathogenesis, Tumorigenesis, and Effects of Host Immunity in Space",Spaceflight Study,"Gene expression levels were determined in 3rd instar and adult Drosophila melanogaster reared during spaceflight, to elucidate the genetic and molecular mechanisms underpinning the effects of microgravity on the immune system. The goal was to validate the Drosophila model for understanding alterations of innate immune responses in humans due to spaceflight. Five containers of flies, with ten female and five male fruit flies in each container, were housed and bred on the space shuttle (average orbit altitude of 330.35 km) for 12 days and 18.5 hours, with a new generation reared in microgravity. RNA was extracted on the day of shuttle landing from whole body animals (3rd instar larvae and adults), hybridized to Drosophila 2.0 Affymetrix genome arrays, and the expression level of all genes was normalized against the gene expression level from the corresponding developmental stage animals raised on ground. Spaceflight altered the expression of larval genes involved in the maturation of plasmatocytes (macrophages) and their phagocytic response, as well as the level of constitutive expression of pattern recognition receptors and opsonins that specifically recognize bacteria, and of lysozymes, antimicrobial peptide pathway and immune stress genes, hallmarks of humoral immunity. Larval microarrays (FL 6 samples) are based on RNA extracted from 6 independent sets of 50 mid 3rd instar larvae reared in microgravity and collected on the day of landing after 12 days and 18.5 hours on the space shuttle and the same number of control larvae raised on ground (GL 6 samples). Adults microarrays (F1 3 samples) are based on RNA from 3 sets of 20 adult females each, that emerged during spaceflight and within 4 hours of landing and the same number of adult females from the corresponding ground control containers (G1 3 samples).",Drosophila melanogaster,7227,,,
2,OSD-4,OSD-4,The Effects of Vector-Averaged Gravity on the Development of T cells,Spaceflight Study,"Microarray Analysis of Space-flown Murine Thymus Tissue Reveals Changes in Gene Expression Regulating Stress and Glucocorticoid Receptors. We used microarrays to detail the gene expression of space-flown thymic tissue and identified distinct classes of up-regulated genes during this process. We report here microarray gene expression analysis in young adult C57BL/6NTac mice at 8 weeks of age after exposure to spaceflight aboard the space shuttle (STS-118) for a period of 13 days. Upon conclusion of the mission, thymus lobes were extracted from space flown mice (FLT) as well as age- and sex-matched ground control mice 

## Create Missions-CONDUCTED_MIcS-Study Relationships

In [13]:
mission_conducted_study = metadata[["mission_id", "identifier"]]
# Not all studies have an associated mission (e.g., ground studies)
mission_conducted_study = mission_conducted_study[mission_conducted_study["mission_id"] != ""].copy()
mission_conducted_study.rename(columns={"mission_id": "from", "identifier": "to", }, inplace=True)

In [14]:
mission_conducted_study_rels = gl.save_dataframe_to_kg(mission_conducted_study, 'Mission-CONDUCTED_MIcS-Study', rel_dir)
print(f"Number of Mission-CONDUCTED_MIcS-Study relationships: {mission_conducted_study_rels.shape[0]}")
mission_conducted_study_rels.head()

Number of Mission-CONDUCTED_MIcS-Study relationships: 132


,from,to
0,STS-121,OSD-1
1,STS-121,OSD-3
4,Expedition-14,OSD-13
6,STS-135,OSD-25
9,Soyuz-TMA-2,OSD-36
